# 09r How big should the personal map be? (THINGS-EEG2)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 01 and 03b, and the THINGS-EEG2 dataset `cross-subject-brain-encoding-eeg` (09g).

Development analysis, not a prespecified test. Every map size on one footing: none (borrowing), per-sensor scaling per window, the 17 x 17 map truncated to
rank 1, 2, 4, 8, the full map per 50, 100, 200 ms window or for the whole response, and the low-rank map (K = 25 to
400); k = 50 to 1,000. Scores: encoding r at 100 to 200 ms, centred decoding, and raw decoding with the person's
labelled mean.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09r-mapsize', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"ks": [50, 100, 250, 500, 1000], "draws": [13, 14, 15], "ranks": [1, 2, 4, 8], "windows_ms": [50, 100, 200], "components": [25, 50, 100, 200, 400], "window_s": [0.1, 0.2], "folds": 5, "seed": 356}')
PEOPLE = ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10']
TAG = '09r'

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.utils import config as C

import shutil
from cross_subject_encoding.data.splits import subset
with np.load(C.find_artifact("paper_alexnet_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
subsets = json.loads(C.find_artifact(C.EEG_K_SUBSETS).read_text())
people = PEOPLE
ATTACHED = te.attached(C.EEG_RAW)  # the 09g dataset (flat names) or the release layout
WORK = C.EEG_RAW if ATTACHED else Path("/kaggle/tmp/things_eeg2")
y_train, bio_test = {}, {}
for s in people:
    if not ATTACHED and te.subject_file(WORK, s, "test") is None:
        te.download_subject(s, WORK)
    tr = te.load_part(te.subject_file(WORK, s, "training"))
    y_train[s], times = tr["data"].mean(1).astype(np.float32), tr["times"]
    bio_test[s] = te.load_part(te.subject_file(WORK, s, "test"))["data"].astype(np.float32)
    del tr
    if not ATTACHED:
        shutil.rmtree(WORK / s, ignore_errors=True)
order = lambda s, d, k: np.asarray(subset(subsets, s, d, k))
print(len(people), 'people loaded |', f'{(time.monotonic() - T0) / 60:.1f} min')

In [ ]:
n_ch, n_t = 17, len(times)
N_TRAIN = len(next(iter(y_train.values())))
win = ha.window_index(times, 100)
t3 = lambda a, n: np.asarray(a, np.float64).reshape(n, n_ch, n_t)
post = np.repeat((times >= -1e-9)[None, :], n_ch, 0).ravel()

def sources(ztr, zte):
    tr = np.empty((len(people), len(ztr), n_ch * n_t), np.float32)
    te_ = np.empty((len(people), len(zte), n_ch * n_t), np.float32)
    q = np.concatenate([ztr, zte])
    for i, s in enumerate(people):
        pred = pl.ols_paper(ztr, y_train[s].reshape(len(ztr), -1), q)
        tr[i], te_[i] = pred[:len(ztr)], pred[len(ztr):]
    return tr, te_

def similarity(pred, meas):
    # centred over the 200 test images, then Pearson over the post-onset features
    p, m = pred[:, post] - pred[:, post].mean(0), meas[:, post] - meas[:, post].mean(0)
    p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
    m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
    return m @ p.T / p.shape[1]  # (measured, candidates)

def calibrate(r, how):
    if how == "none":
        return r
    if how == "csls":  # cross-domain similarity local scaling (Conneau et al. 2018), k = 10, as SATTC's geometric expert
        rm = np.sort(r, 1)[:, -10:].mean(1)
        rp = np.sort(r, 0)[-10:, :].mean(0)
        return 2 * r - rm[:, None] - rp[None, :]
    if how == "dual_z":  # each candidate's similarities standardized over the test responses, then each response's
        z = (r - r.mean(0)) / (r.std(0) + 1e-30)
        return (z - z.mean(1, keepdims=True)) / (z.std(1, keepdims=True) + 1e-30)
    raise ValueError(how)

def test_time_map(pred, meas):
    # label-free, transductive (the 200 test responses as a batch, no pairing used): per 100 ms window, re-mix the
    # prediction's channels so that their covariance over images and time points matches the measured responses'
    from scipy.linalg import sqrtm
    p3, m3 = t3(pred, len(pred)), t3(meas, len(meas))
    out = p3.copy()
    for w in np.unique(win):
        cols = win == w
        xp = (p3[:, :, cols] - p3[:, :, cols].mean(0)).transpose(0, 2, 1).reshape(-1, n_ch)
        xm = (m3[:, :, cols] - m3[:, :, cols].mean(0)).transpose(0, 2, 1).reshape(-1, n_ch)
        cp = np.cov(xp, rowvar=False) + 1e-6 * np.eye(n_ch)
        cm = np.cov(xm, rowvar=False) + 1e-6 * np.eye(n_ch)
        a = np.real(np.linalg.inv(sqrtm(cp)) @ sqrtm(cm))
        mean = p3[:, :, cols].mean(0)
        out[:, :, cols] = np.einsum("nct,cd->ndt", p3[:, :, cols] - mean, a) + mean
    return out.reshape(len(pred), -1)

def encoding_r(pred, meas):
    # per-output correlation across the 200 test images with the repetition average, 100-200 ms, averaged
    sel = np.repeat(((times >= 0.1) & (times < 0.2))[None, :], n_ch, 0).ravel()
    p, m = pred[:, sel] - pred[:, sel].mean(0), meas[:, sel] - meas[:, sel].mean(0)
    return float(np.mean((p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)))

def accuracy(r):
    rank = (r > np.diag(r)[:, None]).sum(1)
    return float((rank < 1).mean() * 100), float((rank < 5).mean() * 100)

def ci(x):
    se = x.std(ddof=1) / np.sqrt(len(x))
    t = stats.t.ppf(0.975, len(x) - 1)
    return {"mean": float(x.mean()), "lower95": float(x.mean() - t * se), "upper95": float(x.mean() + t * se), "n": int(len(x))}

MAP_ALPHAS = h4b.ALPHAS
N_TEST = len(z_test)
src_train, src_test = sources(z_train, z_test)
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
wins = {**{f"w{ms}": ha.window_index(times, ms) for ms in FIXED["windows_ms"]}, "wepoch": np.zeros(n_t, int)}
n_win = {name: len(np.unique(w)) for name, w in wins.items()}

def fit_diag(a_k, y_k, w_idx, seed):
    # per-sensor scaling per window: residual = b * borrowed, one b per sensor and window, ridge by 5-fold CV over images
    k = len(a_k)
    split = np.array_split(np.random.default_rng(seed).permutation(k), FIXED["folds"])
    out = {}
    for w in np.unique(w_idx):
        x, r = a_k[:, :, w_idx == w], y_k[:, :, w_idx == w] - a_k[:, :, w_idx == w]
        x, r = x - x.mean(0), r - r.mean(0)
        sxx, sxr = (x ** 2).sum(2), (x * r).sum(2)  # (images, sensors)
        err = np.zeros((len(MAP_ALPHAS), x.shape[1]))
        for held in split:
            tr = np.setdiff1d(np.arange(k), held)
            for j, al in enumerate(MAP_ALPHAS):
                b = sxr[tr].sum(0) / (sxx[tr].sum(0) + al)
                err[j] += ((r[held] - b[None, :, None] * x[held]) ** 2).sum((0, 2))
        al = np.asarray(MAP_ALPHAS)[err.argmin(0)]
        out[int(w)] = sxr.sum(0) / (sxx.sum(0) + al)
    return out

def apply_diag(b, a, w_idx):
    out = np.array(a, dtype=np.float64, copy=True)
    for w, bw in b.items():
        out[:, :, w_idx == w] += bw[None, :, None] * a[:, :, w_idx == w]
    return out

def similarity_raw(pred, meas):
    p, m = pred[:, post], meas[:, post]
    p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
    m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
    return m @ p.T / p.shape[1]

params = {"avg": 0, "diag_w100": n_win["w100"] * n_ch}
params.update({f"rank{r}_w100": n_win["w100"] * r * (2 * n_ch - r) for r in FIXED["ranks"]})
params.update({f"full_{name}": n_win[name] * n_ch * n_ch for name in wins})
params.update({f"lowrank{K}": K * n_ch * n_t + n_ch * n_t for K in FIXED["components"]})
rows = []
for i, person in enumerate(people):
    t = time.monotonic()
    others = [j for j in range(len(people)) if j != i]
    avg_train, avg_test = src_train[others].mean(0).astype(np.float64), src_test[others].mean(0).astype(np.float64)
    avg_t3 = t3(avg_test, N_TEST)
    bases = {K: lm.prediction_basis(avg_train, K) for K in FIXED["components"]}
    meas = bio_test[person].mean(1).reshape(N_TEST, -1).astype(np.float64)
    preds, on_k = {"avg|0|-": avg_t3}, {}
    for d in FIXED["draws"]:
        for k in FIXED["ks"]:
            idx = order(person, d, k)
            y_k = np.asarray(y_train[person][idx], np.float64)
            a_k = t3(avg_train[idx], k)
            def put(name, test, train):
                preds[f"{name}|{k}|{d}"], on_k[f"{name}|{k}|{d}"] = test, (train, y_k)
            b = fit_diag(a_k, y_k, wins["w100"], d)
            put("diag_w100", apply_diag(b, avg_t3, wins["w100"]), apply_diag(b, a_k, wins["w100"]))
            for name, w_idx in wins.items():
                coefs = h4b.fit_map(a_k, y_k, w_idx, seed=d)[0]
                put(f"full_{name}", h4b.apply_map(coefs, avg_t3, w_idx), h4b.apply_map(coefs, a_k, w_idx))
                if name == "w100":
                    for r in FIXED["ranks"]:  # truncated, as the 09b/09c ablation
                        cr = ha.truncate_rank(coefs, r)
                        put(f"rank{r}_w100", h4b.apply_map(cr, avg_t3, wins["w100"]), h4b.apply_map(cr, a_k, wins["w100"]))
            for K, (mean, basis) in bases.items():
                model = lm.fit_lowrank_map(avg_train[idx], y_k.reshape(k, -1), mean, basis, seed=d)
                put(f"lowrank{K}", t3(lm.apply_lowrank_map(model, avg_test, mean, basis), N_TEST),
                    t3(lm.apply_lowrank_map(model, avg_train[idx], mean, basis), k))
    scored = paper_correlation(preds, bio_test[person], n_iter=100)
    for name, curve in scored["correlation"].items():
        method, k, d = name.split("|")
        pred = np.asarray(preds[name]).reshape(N_TEST, -1)
        row = {"person": person, "method": method, "params": params[method], "k": k, "draw": d,
               "r_100_200ms": float(curve[:, window].mean()), "top1_centred": accuracy(similarity(pred, meas))[0]}
        if name in on_k:
            train, y_k = on_k[name]
            fixed = nc.with_labelled_mean(pred, np.asarray(train).reshape(len(y_k), -1), y_k.reshape(len(y_k), -1))
            row["top1_raw_labelled_mean"] = accuracy(similarity_raw(fixed, meas))[0]
        else:
            row["top1_raw_labelled_mean"] = accuracy(similarity_raw(pred, meas))[0]  # k = 0: plain raw score
        rows.append(row)
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "09r_scores.csv", index=False)
    print(person, f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
frame = pd.DataFrame(rows)
def per_person(method, k, metric):
    f = frame[(frame.method == method) & (frame.k == str(k))]
    return f.groupby("person")[metric].mean().reindex(people).to_numpy()
metrics = ("r_100_200ms", "top1_centred", "top1_raw_labelled_mean")
avg = {m: per_person("avg", 0, m) for m in metrics}
methods = [m for m in params if m != "avg"]
summary = {"post_hoc": "development, descriptive; THINGS-EEG2, held-out person", "params": params,
           "avg": {m: float(avg[m].mean()) for m in metrics}, "by_k": {}}
for k in FIXED["ks"]:
    summary["by_k"][k] = {m: {met: {"mean": float(per_person(m, k, met).mean()), "gain": ci(per_person(m, k, met) - avg[met])}
                              for met in metrics} for m in methods}
    best = max(methods, key=lambda m: summary["by_k"][k][m]["r_100_200ms"]["mean"])
    summary["by_k"][k]["best_encoding"] = best
(C.METRICS_DIR / "09r_summary.json").write_text(json.dumps(summary, indent=2))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
colours = plt.cm.viridis(np.linspace(0.1, 0.9, len(FIXED["ks"])))
order_m = sorted(methods, key=lambda m: params[m])
x = np.array([params[m] for m in order_m])
for a, met, label in ((ax[0], "r_100_200ms", "encoding gain over others' average (r)"), (ax[1], "top1_raw_labelled_mean", "decoding top-1 of 200 (%), raw")):
    for k, c in zip(FIXED["ks"], colours):
        y = [summary["by_k"][k][m][met]["gain"]["mean"] if met == "r_100_200ms" else summary["by_k"][k][m][met]["mean"] for m in order_m]
        a.plot(x, y, "o-", color=c, ms=3, lw=1.2, label=f"k = {k}")
    a.set(xscale="log", xlabel="free weights in the map", ylabel=label)
    if met == "r_100_200ms":
        a.axhline(0, color="black", lw=0.6)
    else:
        a.axhline(summary["avg"][met], color="#7A7A7A", ls="--", lw=1)
ax[0].legend(fontsize=7)
ax[0].set_title("09r: map size against labelled images")
fig.tight_layout()
fig.savefig(C.METRICS_DIR / "09r_figure.png", dpi=150)
print(json.dumps({k: {m: round(v["r_100_200ms"]["gain"]["mean"], 4) for m, v in b.items() if isinstance(v, dict)} for k, b in summary["by_k"].items()}, indent=1))
print({k: b["best_encoding"] for k, b in summary["by_k"].items()})